In [2]:
# Cell 1 — imports and connection
import os
import json
import pandas as pd
from dotenv import load_dotenv
import snowflake.connector
from openai import OpenAI
 
load_dotenv()
 
conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database='ANALYTICS_PROD',
    schema='PUBLIC',
)
 
def run_query(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    cur.close()
    return pd.DataFrame(rows, columns=cols)
 
openai_client = OpenAI(api_key=os.getenv("OPENAI_KEY"))
 
print("Connected.")

Connected.


In [11]:
# ── Cell 2 — pull ALL currently-true postings by job_id ────────────────────────
df = run_query("""
    SELECT job_id, job_title, source, description,
           explicitly_encourages_applicants AS old_flag
    FROM ANALYTICS_PROD.PUBLIC.FCT_JOB_POSTINGS
    WHERE explicitly_encourages_applicants = TRUE
""")
df.columns = [c.lower() for c in df.columns]
 
print(f"Pulled {len(df)} currently-TRUE postings:")
print(df[['job_id', 'job_title', 'source']].to_string())

Pulled 24 currently-TRUE postings:
                      job_id                                                                 job_title      source
0   6vi0NY-dMVCh0ld7AAAAAA==                                                [P] Data Scientist, Policy     jsearch
1                  703117190                            Analytics Engineer, Service Ops Analytics & AI  theirstack
2                  709041443                    Data Analyst (FGP) - Manhattan, Central Billing Office  theirstack
3                    9720468                                           Data Analyst - Data & Analytics     builtin
4                    9593070                         Data Analyst - Economic Insights & Communications     builtin
5   YnVw2gSyafgb-m6FAAAAAA==                                            Data Analyst - Full-Time Roles     jsearch
6   R0IfExR1nrxN_kQyAAAAAA==                                          Data Analyst - Immediate Opening     jsearch
7                  721493914                 

In [12]:
# ── Cell 3 — the updated prompt ────────────────────────────────────────────────
# Paste your FULL updated job_extraction.txt content here (the whole file,
# not just the one field) so the test matches what the real pipeline will run.
 
UPDATED_PROMPT = """
You are a job posting analyst specializing in data roles.
Given a job title and description, extract structured metadata.

Return ONLY a valid JSON object — no preamble, no markdown, no explanation.
The JSON must conform to this exact schema:

{
  "inferred_seniority": <"entry" | "mid" | "senior">,
  "title_seniority_signal": <"accurate" | "overstated" | "understated">,
  "title_signal_reasoning": <string — 1-2 sentences or null if accurate>,
  "role_archetype": <"data_analyst" | "analytics_engineer" | "data_engineer" | "hybrid" | "software_engineer">,
  "work_focus": <string — short phrase describing primary day-to-day work, max 8 words>,
  "tech_stack_required": <array of lowercase strings — tools explicitly required>,
  "tech_stack_preferred": <array of lowercase strings — tools listed as preferred or nice to have>,
  "paradigms_required": <array of lowercase strings — concepts/practices explicitly required>,
  "paradigms_preferred": <array of lowercase strings — concepts/practices listed as preferred>,
  "degree_requirement": <"none" | "bachelors" | "masters" | "equivalent_ok">,
  "years_required_min": <integer or null>,
  "years_required_max": <integer or null>,
  "salary_min": <integer or null>,
  "salary_max": <integer or null>,
  "acknowledges_ai": <true | false>,
  "domain": <string or null>,
  "explicitly_encourages_applicants": <true | false>,
  "confidence_score": <float 0.0–1.0>
}

Rules:

inferred_seniority: base this on actual requirements, NOT the title.
  entry = 0-2 years or no experience required
  mid = 2-5 years
  senior = 5+ years

title_seniority_signal: compare the seniority implied by the job title against inferred_seniority.
  accurate = title aligns with inferred seniority
  overstated = title implies higher seniority than inferred_seniority supports
  understated = title implies lower seniority than inferred_seniority supports

title_signal_reasoning: 1-2 sentences explaining the mismatch. null if title_seniority_signal is "accurate".

role_archetype: the primary operational reality of the role.
  data_analyst = focused on querying, reporting, and business insights
  analytics_engineer = focused on data modeling, transformation, and serving clean data to analysts
  data_engineer = focused on building and maintaining pipelines and infrastructure
  hybrid = genuinely split between two or more data role types with no clear primary
  software_engineer = primarily a software engineering role with minimal data focus

work_focus: a short free-text phrase (max 8 words) describing what the person in this
  role actually does day to day. Be specific and concrete.
  Good examples:
    "build and maintain ELT pipelines for analytics"
    "create executive dashboards and ad hoc reports"
    "design dimensional models in dbt for analysts"
    "analyze user behavior to inform product decisions"
  Bad examples:
    "data work" (too vague)
    "various data engineering and analytics tasks" (too generic)

tech_stack_required vs tech_stack_preferred:
  required = explicitly listed under required qualifications or minimum qualifications
  preferred = explicitly listed under preferred, nice to have, or bonus qualifications
  if the posting does not distinguish, put all tools in tech_stack_required
  normalize names: "MS Excel" -> "excel", "Google BigQuery" -> "bigquery", "Apache Airflow" -> "airflow"
  always use "looker" not "lookers" — it is a product name, not a plural noun

paradigms_required vs paradigms_preferred:
  paradigms are concepts and practices, not tools — e.g. "dimensional modeling", "etl design",
  "data governance", "data warehousing", "pipeline orchestration", "statistical analysis",
  "data modeling", "ml pipelines", "data quality", "data lakes"
  apply the same required vs preferred split as tech stack
  if the posting does not distinguish, put all paradigms in paradigms_required

degree_requirement:
  none = no degree mentioned or explicitly not required
  bachelors = bachelor's degree explicitly required
  masters = master's degree explicitly required or strongly preferred
  equivalent_ok = degree mentioned but equivalent experience explicitly accepted

years_required_min / years_required_max:
  parse from required qualifications only — ignore preferred qualifications
  if a single number is given, set min = max = that number
  if no years are mentioned in required qualifications, set both to null

salary_min / salary_max:
  extract only when explicitly stated in the posting as a number
  always as annual integer (convert hourly if needed: hourly * 2080)
  null if not mentioned

acknowledges_ai: true if the posting explicitly mentions AI, LLMs, machine learning tools,
  ChatGPT, Copilot, or similar in the context of the role or company's work.
  false if no mention of AI or ML whatsoever.

domain: the industry vertical the company operates in. Use a short lowercase string.
  Common values: "finance", "healthcare", "retail", "tech", "consulting", "government",
  "media", "real estate", "education", "logistics"
  null if the role is at a general-purpose tech company or domain is not determinable.

explicitly_encourages_applicants: true ONLY if the posting explicitly invites candidates
  who do not fully meet the listed qualifications, requirements, or experience level to
  apply anyway — a SKILLS/EXPERIENCE GAP statement specifically.

  CRITICAL EXCLUSION: Generic equal-opportunity-employer language is NEVER this signal,
  even though it also uses the words "encourage" and "apply". If the only "encourage to
  apply" language in the posting is part of an EEO/equal-opportunity statement (e.g.
  "all qualified individuals are encouraged to apply", "we are an equal opportunity
  employer and encourage applications from all backgrounds"), the answer is false.
  This is true regardless of how that sentence is worded — it is about legal compliance
  and demographic inclusion, never about a candidate's qualifications.

  ALSO NOT SUFFICIENT: generic enthusiasm or culture language with no qualifications
  gap named (e.g. "if you're passionate about this work and ready for a challenging
  role, we encourage you to apply"). Passion, attitude, or excitement are not
  qualifications — this is false unless a specific gap is also named.

  Read the ENTIRE posting, not just sentences containing "encourage" or "invite". The
  qualifications-gap statement does not need to appear in the same sentence as those
  words. For example, "we welcome candidates from all backgrounds, including recent
  graduates without prior technical experience" is true even without the word
  "encourage" appearing nearby — it explicitly names a real qualifications gap
  (no prior technical experience) and says those candidates are welcome.

  true examples (the actual signal — a qualifications/experience gap):
    "we encourage you to apply even if you don't meet every qualification"
    "if you don't match every criteria, we encourage you to apply"
    "if your experience doesn't align perfectly with the job description, apply anyway"
    "not everyone will meet all the qualifications on day one — apply anyway"
    "we welcome recent graduates without prior technical experience"

  false examples (look similar, but are NOT this signal):
    "all qualified individuals are encouraged to apply" (EEO boilerplate — ALWAYS false)
    "we encourage candidates of all backgrounds to apply" (diversity, not qualifications)
    "we are an equal opportunity employer" (legal boilerplate)
    "reliable and detail-oriented candidates are encouraged to apply" (generic, no gap stated)
    "if you're passionate about this work and ready for a challenging role, apply" (enthusiasm only, no gap stated)

  The word "encourage" alone is NEVER sufficient. The posting must name or imply a
  GAP between the candidate's actual qualifications and the listed requirements,
  somewhere in the text. When in doubt, default to false. This field should be rare.

confidence_score: your confidence that the extraction is accurate given description quality.
  penalize heavily for vague, templated, or very short descriptions
"""

In [15]:
job_id = '703117190'
row = df[df['job_id'] == job_id].iloc[0]

response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[
        {"role": "system", "content": UPDATED_PROMPT + "\n\nIMPORTANT: After the JSON object, on a new line, explain in 1-2 sentences why you set explicitly_encourages_applicants to true or false."},
        {"role": "user", "content": f"Job Title: {row['job_title']}\n\nDescription:\n{row['description']}"},
    ],
    timeout=30,
)
print(response.choices[0].message.content)

{
  "inferred_seniority": "mid",
  "title_seniority_signal": "accurate",
  "title_signal_reasoning": null,
  "role_archetype": "data_engineer",
  "work_focus": "design and maintain scalable data pipelines",
  "tech_stack_required": [
    "sql",
    "python",
    "dbt",
    "snowflake",
    "git",
    "airflow"
  ],
  "tech_stack_preferred": [],
  "paradigms_required": [
    "etl design",
    "data governance",
    "data quality",
    "data modeling",
    "ci/cd"
  ],
  "paradigms_preferred": [],
  "degree_requirement": "none",
  "years_required_min": null,
  "years_required_max": null,
  "salary_min": null,
  "salary_max": null,
  "acknowledges_ai": true,
  "domain": "insurance",
  "explicitly_encourages_applicants": true,
  "confidence_score": 0.9
}

The posting explicitly invites candidates who may not meet every qualification to apply, indicating a qualifications gap. This is evident in the phrase "if you are passionate about data engineering and ready to take on a challenging, impa

In [13]:
# ── Cell 4 — re-run the new prompt against just these postings ────────────────
def test_one(job_title: str, description: str) -> dict:
    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": UPDATED_PROMPT},
                {"role": "user", "content": f"Job Title: {job_title}\n\nDescription:\n{description}"},
            ],
            timeout=30,  # seconds
        )
    except Exception as e:
        print(f"  ⚠️ Request failed/timed out for '{job_title}': {e}")
        return None

    raw = response.choices[0].message.content
    try:
        parsed = json.loads(raw)
        return parsed.get("explicitly_encourages_applicants")
    except json.JSONDecodeError:
        print(f"  ⚠️ Could not parse response for '{job_title}': {raw[:200]}")
        return None
 
print("\n=== RESULTS ===")
new_flags = []
for _, row in df.iterrows():
    new_flag = test_one(row["job_title"], row["description"] or "")
    new_flags.append(new_flag)
    if new_flag == False:
        status = "FLIPPED → false"
    elif new_flag == True:
        status = "STAYED true"
    else:
        status = "— (unexpected)"
    print(f"{status:18s} | {row['job_title']} ({row['source']}) | job_id={row['job_id']}")
 
df["new_flag"] = new_flags


=== RESULTS ===
STAYED true        | [P] Data Scientist, Policy (jsearch) | job_id=6vi0NY-dMVCh0ld7AAAAAA==
STAYED true        | Analytics Engineer, Service Ops Analytics & AI (theirstack) | job_id=703117190
FLIPPED → false    | Data Analyst (FGP) - Manhattan, Central Billing Office (theirstack) | job_id=709041443
STAYED true        | Data Analyst - Data & Analytics (builtin) | job_id=9720468
STAYED true        | Data Analyst - Economic Insights & Communications (builtin) | job_id=9593070
STAYED true        | Data Analyst - Full-Time Roles (jsearch) | job_id=YnVw2gSyafgb-m6FAAAAAA==
STAYED true        | Data Analyst - Immediate Opening (jsearch) | job_id=R0IfExR1nrxN_kQyAAAAAA==
STAYED true        | Data Analyst (theirstack) | job_id=721493914
STAYED true        | Data Analyst (builtin) | job_id=9564259
STAYED true        | Data Analyst (builtin) | job_id=9547634
STAYED true        | Data Analyst (builtin) | job_id=9536284
STAYED true        | Data Analyst (builtin) | job_id=9623166
F

In [14]:
print(UPDATED_PROMPT[:500])
print("...")
print("ALSO NOT SUFFICIENT" in UPDATED_PROMPT)


You are a job posting analyst specializing in data roles.
Given a job title and description, extract structured metadata.

Return ONLY a valid JSON object — no preamble, no markdown, no explanation.
The JSON must conform to this exact schema:

{
  "inferred_seniority": <"entry" | "mid" | "senior">,
  "title_seniority_signal": <"accurate" | "overstated" | "understated">,
  "title_signal_reasoning": <string — 1-2 sentences or null if accurate>,
  "role_archetype": <"data_analyst" | "analytics_eng
...
True


In [16]:
# ── Cell 5 — summary + which job_ids need updating in Snowflake ───────────────
flipped = df[df["new_flag"] == False]
stayed_true = df[df["new_flag"] == True]
 
print(f"\nTotal currently-true postings tested: {len(df)}")
print(f"Flipped to false (corrected):          {len(flipped)}")
print(f"Stayed true (should be legit):          {len(stayed_true)}")
 
print("\n--- Stayed TRUE — read these to confirm they're legit qualifications-gap statements ---")
for _, row in stayed_true.iterrows():
    print(f"  {row['job_title']} ({row['source']}) — job_id={row['job_id']}")
 
print("\n--- Flipped to FALSE — these are the corrected job_ids ---")
flipped_ids = flipped["job_id"].tolist()
print(flipped_ids)


Total currently-true postings tested: 24
Flipped to false (corrected):          2
Stayed true (should be legit):          22

--- Stayed TRUE — read these to confirm they're legit qualifications-gap statements ---
  [P] Data Scientist, Policy (jsearch) — job_id=6vi0NY-dMVCh0ld7AAAAAA==
  Analytics Engineer, Service Ops Analytics & AI (theirstack) — job_id=703117190
  Data Analyst - Data & Analytics (builtin) — job_id=9720468
  Data Analyst - Economic Insights & Communications (builtin) — job_id=9593070
  Data Analyst - Full-Time Roles (jsearch) — job_id=YnVw2gSyafgb-m6FAAAAAA==
  Data Analyst - Immediate Opening (jsearch) — job_id=R0IfExR1nrxN_kQyAAAAAA==
  Data Analyst (theirstack) — job_id=721493914
  Data Analyst (builtin) — job_id=9564259
  Data Analyst (builtin) — job_id=9547634
  Data Analyst (builtin) — job_id=9536284
  Data Analyst (builtin) — job_id=9623166
  Data Engineer (theirstack) — job_id=733446081
  Data Engineer (theirstack) — job_id=733240677
  Data Engineer (theirst

In [18]:
# ── Cell 6 — if everything looks right, write the corrections directly ────────
# This updates JOB_ENRICHMENT for just these job_ids — no full re-enrichment needed,
# and no need to truncate the table at all.
#
# IMPORTANT: only run this cell after reading Cell 5's output and confirming the
# flipped and stayed-true buckets both look correct.
 
flipped_ids = ['709041443', '704438351']

for job_id in flipped_ids:
    cur = conn.cursor()
    cur.execute(f"""
        UPDATE ENRICHED.PUBLIC.JOB_ENRICHMENT
        SET EXPLICITLY_ENCOURAGES_APPLICANTS = FALSE
        WHERE JOB_ID = '{job_id}'
    """)
    cur.close()
conn.commit()
print(f"Updated {len(flipped_ids)} rows in JOB_ENRICHMENT.")

Updated 2 rows in JOB_ENRICHMENT.
